# Phase-Space Catastrophes & Pseudo-Differential Dynamics: A 1D Lagrangian Analysis

---

## Executive Theoretical Overview

In wave mechanics and microlocal analysis, **caustics** denote spatial regions where the classical ray approximation predicts infinite energy density. Geometrically, an oscillatory integral of the form:

$$u(x) = \int_{\mathbb{R}} a(x, \xi) \, \mathrm{e}^{\mathrm{i}\phi(x, \xi)} \, \mathrm{d}\xi$$

concentrates its amplitude along the **stationary phase set** $\Sigma_{\phi} = \{(x, \xi) \in \mathbb{R} \times \mathbb{R} : \partial_{\xi}\phi(x, \xi) = 0\}$.

The set $\Sigma_{\phi}$ parameterizes a 1D **Lagrangian submanifold** $\mathcal{L} \subset T^*\mathbb{R}$ embedded in phase space $(x, \xi)$. A caustic occurs precisely at the singular points of the canonical projection $\pi: \mathcal{L} \to \mathbb{R}$ onto physical coordinate space $x$:

$$\mathcal{C} = \left\{ (x, \xi) \in \Sigma_{\phi} \;:\; \frac{\partial^2 \phi}{\partial \xi^2}(x, \xi) = 0 \right\} \quad \iff \quad \frac{\mathrm{d}x}{\mathrm{d}\xi} = 0$$

At these critical points, the phase projection turns vertical in $(x, \xi)$, causing adjacent rays to focus and triggering higher-order asymptotic scalings governed by Arnold’s $A_k$ singularity classification.

This notebook explores the interaction between these singular phase-space structures and **pseudo-differential operators** ($\psi\mathrm{Op}$) defined via spatial and momentum-dependent symbols $S(x, \xi)$:

$$[\psi\mathrm{Op}(S) u](x) = \frac{1}{2\pi} \int_{\mathbb{R}} S(x, \xi) \, \hat{u}(\xi) \, \mathrm{e}^{\mathrm{i} x \xi} \, \mathrm{d}\xi$$

By employing SciPy's centered Short-Time Fourier Transform (STFT), we visualize the phase-space manifolds $\xi(x)$ in real time directly beneath physical wavefield dynamics $[u(x), (\psi\mathrm{Op}(S)u)(x)]$.

---

### The Boundary of Linear Asymptotics vs. Autonomous Nonlinear PDEs

Within the framework of linear oscillatory integrals and asymptotic wave theory, wavefields are strictly governed by the principle of linear superposition. Mathematically, the phase function $\phi(x, \xi)$ dictating the geometry of the Lagrangian manifold $\mathcal{L}$ is entirely decoupled from the wavefield's amplitude or energy density $\vert{}u(x)\vert{}^2$. Consequently, when two wave packets and their underlying manifolds $\mathcal{L}_1$ and $\mathcal{L}_2$ overlap, they traverse one another without mutual scattering, physical feedback, or energy exchange.

To simulate an "interaction" within an explicit integral framework, one must resort to a **kinematic approach**—injecting exogenous, time-dependent coupling parameters (such as a scripted repulsion barrier $\gamma(t)$) directly into the phase function. While these parameter-coupled systems mimic topological warping, caustic restructuring, and branch distortion, the underlying wavefields do not alter their own medium.

True **autonomous nonlinear interaction**—where wave manifolds mutually deform because the wave fields self-consistently alter each other's governing medium or local phase velocities—requires the phase $\phi(x,t)$ to become a functional of local wave intensity. This dynamics is governed by nonlinear partial differential equations (PDEs):

* **Nonlinear Schrödinger Equation (NLSE):** $\mathrm{i} u_t + u_{xx} + \chi \vert{}u\vert{}^2 u = 0$, where Kerr non-linearity drives self-phase modulation and cross-phase modulation (XPM), causing overlapping manifold branches to dynamically focus, attract, or repel.
* **Korteweg–de Vries Equation (KdV):** $u_t + u u_x + u_{xxx} = 0$, where convective non-linear steepening induces a **gradient catastrophe** ($\partial_x u \to \infty$), regularized by third-order dispersion into a rank-ordered soliton train and a Dispersive Shock Wave (DSW).

---

### Pseudo-Spectral PDE Solver

The PDE solver evolves autonomous equations written in the general operator form:

$$\frac{\partial u}{\partial t} = L u + \mathcal{N}(u)$$

where $L$ is a spatial linear operator and $\mathcal{N}(u)$ represents spatial nonlinear forcing (e.g., $u u_x$ for KdV or $\vert{}u\vert{}^2 u$ for NLSE).

The linear term $L$ can be either a classical **differential operator** or a general **pseudo-differential operator** $\psi\mathrm{Op}(S)$:

* **Differential Operator:** $L = \partial_x^3 \implies \mathbf{L}(k) = \mathrm{i} k^3$. The action transforms into point-wise multiplication in Fourier space: $\mathcal{F}\{ L u \}(k) = \mathbf{L}(k) \hat{u}(k)$.
* **Pseudo-Differential Operator:** $L = \psi\mathrm{Op}(S)$. When the symbol depends on the spatial coordinate $x$, the operator is evaluated in physical space via **Kohn–Nirenberg quantization**:

$$[L u](x) = [\psi\mathrm{Op}(S) u](x) = \frac{1}{2\pi} \int_{\mathbb{R}} S(x, k) \, \hat{u}(k) \, \mathrm{e}^{\mathrm{i} x k} \, \mathrm{d}k$$


If $S(k)$ is purely momentum-dependent (homogeneous medium), this reduces to a diagonal multiplier $\mathbf{L}(k) = S(k)$.

In Fourier space, the system becomes a set of coupled ODEs: $\frac{\partial \hat{u}}{\partial t} = \mathcal{F}\{ L u \} + \hat{\mathbf{N}}(\hat{u})$.

---

### Numerical Extraction of Instantaneous Phase-Space Manifolds

To track dynamically evolving Lagrangian submanifolds $\mathcal{L}(t) = \{(x, \xi(x,t))\}$ directly from numerical wavefield data without relying on static asymptotic phase definitions, we extract the local instantaneous wavenumber $\xi(x,t)$ via field-theoretic phase derivatives across four specialized computation regimes:

#### 1. Complex Wavefields (NLSE / Hydrodynamic Form)

For complex scalar fields $u(x,t) = R(x,t) \mathrm{e}^{\mathrm{i}\theta(x,t)}$, the local momentum field corresponds to the spatial derivative of the phase $\xi(x,t) = \partial_x \theta(x,t)$. To avoid multi-valued branch cuts and phase-unwrapping artifacts at domain boundaries, we evaluate this via the **Madelung hydrodynamic identity**:

$$\xi(x,t) = \operatorname{Im}\left( \frac{\partial_x u(x,t)}{u(x,t)} \right)$$

The spatial derivative $\partial_x u$ is computed spectrally via standard Fourier multiplication $\mathcal{F}^{-1}\left( \mathrm{i} k_x \hat{u}(k_x) \right)$, providing high numerical precision. Energy mask thresholds ($\vert{}u\vert{} > \epsilon \cdot \max\vert{}u\vert{}$) shield low-amplitude vacuum nodes from division singularities.

#### 2. Real Wavefields (KdV / Analytic Signal Reconstruction)

Because real-valued scalar fields $u(x,t) \in \mathbb{R}$ lack an explicit phase, direct gradient evaluation is undefined. We reconstruct the complex phase-space representation by forming the **analytic signal** $u_a(x,t)$ via the Hilbert transform $\mathcal{H}$:

$$u_a(x,t) = u(x,t) + \mathrm{i} \mathcal{H}[u(x,t)] = R(x,t) \mathrm{e}^{\mathrm{i} \theta(x,t)}$$

The instantaneous wavenumber manifold $\xi(x,t)$ is then extracted by spectrally differentiating the unwrapped analytic phase:

$$\xi(x,t) = \partial_x \operatorname{unwrap}\left(\operatorname{arg}\left(u(x,t) + \mathrm{i}\mathcal{H}[u(x,t)]\right)\right)$$

This allows real-valued shock fronts and soliton trains to project as distinct 1D continuous curves over STFT spectrogram backgrounds.

#### 3. Envelope-Smoothed Hydrodynamics (Colliding Beams / Net Momentum)

When opposing wavepackets collide, fast spatial interference fringes arise at twice the carrier wavenumber ($2k_0$), causing standard single-valued phase derivatives to oscillate violently. To extract the macroscopic energy transport and net momentum transfer under nonlinear Cross-Phase Modulation (XPM), we compute the coarse-grained ratio of hydrodynamic momentum flux $J(x,t)$ to probability density $\rho(x,t)$:

$$J(x,t) = \operatorname{Im}\left( u^*(x,t) \partial_x u(x,t) \right), \qquad \rho(x,t) = \vert{}u(x,t)\vert{}^2$$

We smooth both scalar fields spatially using a Gaussian kernel $G_\sigma(x) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{x^2}{2\sigma^2}\right)$ tuned to the spatial fringe scale $\sigma \approx 0.8 \cdot (\pi / k_0)$:

$$\xi_{\mathrm{eff}}(x,t) = \frac{(G_\sigma * J)(x,t)}{(G_\sigma * \rho)(x,t)}$$

This spatial filtering eliminates fast interference ripples while retaining the net hydrodynamic trajectory, exposing ray bending and beam attraction/repulsion as a smooth single-valued effective velocity field.

#### 4. Phase-Reassigned Short-Time Fourier Transform (Multi-Valued / Crossing Branches)

In regions containing overlapping beams, folded caustics, or multi-valued ray manifolds, single-valued functions $\xi(x,t)$ fail fundamentally. To reconstruct the true multi-valued phase-space manifold without spatial blurring, we perform **Phase Reassignment** within the time-frequency domain.

Given a windowed Short-Time Fourier Transform $S_u(x, \xi) = \int_{-\infty}^{\infty} u(x') w(x' - x) \mathrm{e}^{-\mathrm{i}\xi x'} \mathrm{d}x'$, the exact local phase gradient at every time-frequency bin $(x, \xi)$ is computed using the STFT of the spatial derivative $S_{\partial_x u}(x, \xi)$:

$$\hat{\xi}(x, \xi) = \operatorname{Im}\left( \frac{S_{\partial_x u}(x, \xi)}{S_u(x, \xi)} \right)$$

By reassigning spectral energy from the bin center $\xi$ directly to its local phase derivative coordinate $\hat{\xi}(x, \xi)$ wherever the magnitude exceeds a threshold $\vert{}S_u(x, \xi)\vert{} > \epsilon \cdot \max\vert{}S_u\vert{}$, we extract the full multi-valued manifold $\mathcal{L}(t)$ as an explicit point cloud $\{(x, \hat{\xi})\}$. This resolves crossing branches, multi-wavepacket collisions, and nonlinearly deformed caustic arcs.

### 0. Parameters & Core Engine

In [ ]:
from imports import * 
from psiop import PseudoDifferentialOperator
from solver import PDESolver

In [ ]:
# ==========================================
# GRID & WINDOW SETUP
# ==========================================
N, L = 1024, 20.0
x_grid = np.linspace(-L/2, L/2, N, endpoint=False)
dx = x_grid[1] - x_grid[0]
kx = np.fft.fftshift(2 * np.pi * np.fft.fftfreq(N, d=dx))
window = tukey(N, alpha=0.1)

# ==========================================
# DEFAULT OPERATOR
# ==========================================
x_sym, xi_sym = symbols('x xi', real=True)
S_expr = sqrt(1 + xi_sym**2) - 0.03 * 1j * x_sym**2
Op1D = PseudoDifferentialOperator(expr=S_expr, vars_x=[x_sym], mode='symbol')

# ==========================================
# STFT SETUP
# ==========================================
SFT = ShortTimeFFT(gaussian(128, std=16), hop=4, fs=1/dx, scale_to='magnitude', fft_mode='centered')

# ==========================================
# SYMBOLIC MANIFOLD ENGINE
# ==========================================
def create_manifold_evaluator(phi_expr, x_sym, xi_sym, param_syms):
    dphi_dxi = diff(phi_expr, xi_sym)
    x_sol = solve(dphi_dxi, x_sym)[0]
    latex_eq = rf'x(\xi) = {latex(x_sol)}'
    evaluator = lambdify((xi_sym, *param_syms), x_sol, 'numpy')
    return evaluator, x_sol, latex_eq

import numpy as np
from scipy.signal import hilbert

import numpy as np
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d

import numpy as np
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d

def compute_instantaneous_manifold(
    u_field, 
    dx, 
    method='madelung', 
    threshold=0.05, 
    k0=2.0, 
    SFT=None, 
    Lx=None, 
    xi_lim=20.0,
    return_coords=False
):
    """
    Extracts the instantaneous phase-space manifold (x, xi) from a 1D numerical wavefield.

    Parameters:
    -----------
    u_field : np.ndarray
        1D spatial wavefield (complex for NLSE, real for KdV).
    dx : float
        Spatial grid spacing.
    method : str, optional
        1. 'madelung'   : Complex fields (NLSE). Local phase derivative Im((d_x u) / u).
                          Single-valued. Best for single wavepackets or non-colliding beams.
        2. 'hilbert'    : Real fields (KdV). d_x arg(u + i*H[u]) via Hilbert transform.
                          Single-valued. Best for real dispersive shock waves / solitons.
        3. 'envelope'   : Complex fields (NLSE). Envelope-smoothed flux <J> / <rho>.
                          Single-valued. Filters out fast 2*k0 spatial interference fringes 
                          during beam collisions to show net Kerr momentum transfer.
        4. 'reassigned' : Phase-Reassigned STFT. Local phase derivatives in time-frequency bins.
                          Multi-valued. Fully resolves overlapping wavepackets, crossing 
                          branches, caustics, and nonlinear phase gradient bending.
    threshold : float, optional
        Amplitude cutoff relative to the maximum field amplitude. Default is 0.05.
    k0 : float, optional
        Carrier / mean wavenumber. Used when method='envelope' to tune the Gaussian 
        smoothing scale (sigma_pts) for filtering interference fringes. Default is 2.0.
    SFT : ShortTimeFFT instance, optional
        Required when method='reassigned'.
    Lx : float, optional
        Domain length. Default is len(u_field) * dx.
    xi_lim : float or tuple, optional
        Wavenumber bounds for clipping/filtering outliers. Default is 20.0.
    return_coords : bool, optional
        If True, forces all methods to return explicit 1D coordinate arrays (x_pts, xi_pts) 
        ready for scatter plotting. If False, methods 1-3 return (xi_inst, mask).

    Returns:
    --------
    For methods 'madelung', 'hilbert', 'envelope' (when return_coords=False):
        xi_inst : np.ndarray (1D wavenumber array on spatial grid)
        mask    : np.ndarray (bool array of valid energy regions)
    For method 'reassigned' OR when return_coords=True:
        x_pts   : np.ndarray (1D array of spatial coordinates)
        xi_pts  : np.ndarray (1D array of local wavenumbers)
    """
    N = len(u_field)
    if Lx is None:
        Lx = N * dx
        
    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    # =========================================================================
    # METHOD 1: MADELUNG HYDRODYNAMIC FORM (Complex fields, e.g., NLSE)
    # =========================================================================
    if method == 'madelung':
        amp = np.abs(u_field)
        mask = amp > (threshold * np.max(amp))

        u_hat = np.fft.fft(u_field)
        grad_u = np.fft.ifft(1j * kx * u_hat)
        
        xi_inst = np.imag(grad_u / (u_field + 1e-12))
        
        if isinstance(xi_lim, (int, float)):
            xi_inst = np.clip(xi_inst, -xi_lim, xi_lim)

        if not return_coords:
            return xi_inst, mask
        
        x_grid = np.linspace(-Lx / 2.0, Lx / 2.0, N, endpoint=False)
        return x_grid[mask], xi_inst[mask]

    # =========================================================================
    # METHOD 2: HILBERT ANALYTIC SIGNAL (Real fields, e.g., KdV)
    # =========================================================================
    elif method == 'hilbert':
        u_real = np.real(u_field)

        u_analytic = hilbert(u_real)
        amp = np.abs(u_analytic)
        mask = amp > (threshold * np.max(amp))

        phase_unwrapped = np.unwrap(np.angle(u_analytic))
        phase_hat = np.fft.fft(phase_unwrapped)
        xi_inst = np.fft.ifft(1j * kx * phase_hat).real
        
        if isinstance(xi_lim, (int, float)):
            xi_inst = np.clip(xi_inst, -xi_lim, xi_lim)

        if not return_coords:
            return xi_inst, mask

        x_grid = np.linspace(-Lx / 2.0, Lx / 2.0, N, endpoint=False)
        return x_grid[mask], xi_inst[mask]

    # =========================================================================
    # METHOD 3: ENVELOPE-SMOOTHED FLUX (Colliding beams / Net momentum)
    # =========================================================================
    elif method == 'envelope':
        u_hat = np.fft.fft(u_field)
        grad_u = np.fft.ifft(1j * kx * u_hat)

        J = np.imag(np.conj(u_field) * grad_u)  # Hydrodynamic momentum density
        rho = np.abs(u_field)**2                 # Probability / field density

        # Smooth over fast interference spatial period lambda = pi / k0
        fringe_period = (2 * np.pi) / (2 * max(abs(k0), 1e-5))
        sigma_pts = (fringe_period / dx) * 0.8

        J_smooth = gaussian_filter1d(J, sigma=sigma_pts)
        rho_smooth = gaussian_filter1d(rho, sigma=sigma_pts)

        mask = rho_smooth > (threshold * np.max(rho_smooth))

        xi_inst = np.zeros(N)
        xi_inst[mask] = J_smooth[mask] / rho_smooth[mask]
        
        if isinstance(xi_lim, (int, float)):
            xi_inst = np.clip(xi_inst, -xi_lim, xi_lim)

        if not return_coords:
            return xi_inst, mask

        x_grid = np.linspace(-Lx / 2.0, Lx / 2.0, N, endpoint=False)
        return x_grid[mask], xi_inst[mask]

    # =========================================================================
    # METHOD 4: PHASE-REASSIGNED STFT (Multi-valued / Crossing branches)
    # =========================================================================
    elif method == 'reassigned':
        if SFT is None:
            raise ValueError("Method 'reassigned' requires passing a ShortTimeFFT object as `SFT`.")

        u_hat = np.fft.fft(u_field)
        grad_u = np.fft.ifft(1j * kx * u_hat)

        stft_u = SFT.stft(u_field)
        stft_ux = SFT.stft(grad_u)
        spec_mag = np.abs(stft_u)

        # Exact local phase gradient at every time-frequency bin (x, xi)
        with np.errstate(divide='ignore', invalid='ignore'):
            xi_reassigned = np.imag(stft_ux / stft_u)

        x_stft = SFT.t(N) - Lx / 2.0

        cutoff = threshold * np.max(spec_mag)
        valid_mask = (spec_mag > cutoff) & np.isfinite(xi_reassigned)

        if xi_lim is not None:
            if isinstance(xi_lim, (int, float)):
                valid_mask &= (np.abs(xi_reassigned) <= xi_lim)
            elif len(xi_lim) == 2:
                valid_mask &= (xi_reassigned >= xi_lim[0]) & (xi_reassigned <= xi_lim[1])

        # Fully vectorized point extraction (no loops)
        X_mesh = np.tile(x_stft, (spec_mag.shape[0], 1))
        x_pts = X_mesh[valid_mask]
        xi_pts = xi_reassigned[valid_mask]

        return x_pts, xi_pts

    else:
        raise ValueError(
            f"Unknown method '{method}'. Valid options are 'madelung', 'hilbert', 'envelope', or 'reassigned'."
        )

# ==========================================
# FACTORIZED VISUALIZATION ENGINE
# ==========================================
class PhaseSpaceAnimator:
    def __init__(self, x_grid, kx, dx, L, window, SFT, operator, 
                 y_lim_field=(-3.5, 3.5), y_lim_spec=(-4.0, 4.0), cmap='magma'):
        self.x_grid, self.kx, self.dx, self.L = x_grid, kx, dx, L
        self.window, self.SFT, self.operator = window, SFT, operator
        self.N = len(x_grid)
        
        self.fig, (self.ax_field, self.ax_spec) = plt.subplots(
            2, 1, figsize=(10, 7), sharex=True, gridspec_kw={'height_ratios': [1, 1.2]}
        )
        
        self.line_u, = self.ax_field.plot([], [], label=r'Input $u(x)$', color='tab:blue', lw=1.8)
        self.line_pu, = self.ax_field.plot([], [], label=r'Action $[\psi\mathrm{Op}(S) u](x)$', color='tab:orange', linestyle='--', lw=1.6)
        self.ax_field.set_ylabel('Amplitude')
        self.ax_field.set_ylim(y_lim_field)
        self.ax_field.grid(True)
        
        self.im_spec = self.ax_spec.imshow(
            np.zeros((SFT.f_pts, SFT.p_num(self.N))), aspect='auto', origin='lower', cmap=cmap
        )
        self.ax_spec.set_xlabel('Spatial Coordinate $x$')
        self.ax_spec.set_ylabel(r'Local Frequency $\xi$')
        self.ax_spec.set_ylim(y_lim_spec)
        
        self.caustic_lines_f, self.caustic_lines_s = [], []
        self.manifold_lines, self.manifold_configs = [], []
        
        self.ax_field.legend(loc='upper left', fontsize=8)
#        self.ax_spec.legend(loc='upper left', fontsize=8)

    def add_caustic(self, label='Caustic $x_c$', color='red'):
        lf = self.ax_field.axvline(x=-999, color=color, linestyle=':', lw=1.5, label=label)
        ls = self.ax_spec.axvline(x=-999, color=color, linestyle=':', lw=1.5)
        self.caustic_lines_f.append(lf)
        self.caustic_lines_s.append(ls)
        self.ax_field.legend(loc='upper left', fontsize=8)

    def add_manifold(self, eval_func, sol_expr, param_syms, color='cyan', prefix='Manifold'):
        static_latex = rf'{prefix} $x(\xi) = {latex(sol_expr)}$'
        line, = self.ax_spec.plot([], [], color, lw=1.5, label=static_latex)
        self.manifold_lines.append(line)
        self.manifold_configs.append((eval_func, sol_expr, param_syms, color, prefix))
        self.ax_spec.legend(loc='upper left', fontsize=8)

    def _update_frame(self, t_val, compute_u, get_caustics, get_man_params, title_func):
        res = compute_u(self.x_grid, t_val)
        if isinstance(res, tuple):
            u_raw, Pu = res
            u = u_raw * self.window
        else:
            u = res * self.window
            Pu = self.operator.apply(u, self.x_grid, self.kx, clamp=1e6, space_window=False)
            u_raw = res
            
        self.line_u.set_data(self.x_grid, np.real(u_raw))
        self.line_pu.set_data(self.x_grid, np.real(Pu))
        
        spec_mag = np.abs(self.SFT.stft(u))
        t_stft = self.SFT.t(self.N) - self.L/2
        xi_stft = 2 * np.pi * self.SFT.f
        self.im_spec.set_data(spec_mag)
        self.im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
        self.im_spec.set_clim(vmin=0, vmax=spec_mag.max())
        
        caustics = get_caustics(t_val)
        for i, x_c in enumerate(caustics):
            if i < len(self.caustic_lines_f):
                self.caustic_lines_f[i].set_xdata([x_c])
                self.caustic_lines_s[i].set_xdata([x_c])
                
        xi_curve = np.linspace(self.ax_spec.get_ylim()[0], self.ax_spec.get_ylim()[1], 300)
        params_list = get_man_params(t_val)
        for i, (eval_func, sol_expr, param_syms, _, prefix) in enumerate(self.manifold_configs):
            params = params_list[i]
            x_man = eval_func(xi_curve, *params)
            self.manifold_lines[i].set_data(x_man, xi_curve)
            
            subs_dict = {sym: Float(val, 2) for sym, val in zip(param_syms, params)}
            x_sol_num = sol_expr.subs(subs_dict)
            self.manifold_lines[i].set_label(rf'{prefix} $x(\xi) = {latex(x_sol_num)}$')
            
        self.ax_spec.legend(loc='upper left', fontsize=8)
        
        if title_func:
            self.ax_field.set_title(title_func(t_val))
            
        return [self.line_u, self.line_pu, self.im_spec, *self.caustic_lines_f, *self.caustic_lines_s, *self.manifold_lines]

    def animate(self, t_values, compute_u, get_caustics, get_man_params, title_func=None, filename=None, fps=15):
        anim = FuncAnimation(
            self.fig, 
            lambda t: self._update_frame(t, compute_u, get_caustics, get_man_params, title_func), 
            frames=t_values, interval=1000/fps
        )
        if filename:
            anim.save(filename, writer='pillow', fps=fps)
        plt.show()
        return anim

### 1. Pure Airy Fold Transition ($A_2$)

* **Mechanism:** Single $A_2$ fold caustic translated through the spatial domain.
* **Phase Space:** The parabolic Lagrangian manifold $x(\xi) = t - \xi^2$ moves horizontally, showing the fold apex at $\xi = 0$.

In [ ]:
t_sym = symbols('t', real=True)
phi_airy = (1.0 / 3.0) * xi_sym**3 + (x_sym - t_sym) * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_airy, x_sym, xi_sym, [t_sym])

xi_int = np.linspace(-4.0, 4.0, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.1 * xi_int**2)

def compute_airy(x_arr, t_val):
    phase = (1.0 / 3.0) * xi_int[:, None]**3 + (x_arr[None, :] - t_val) * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 2.0

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-2.5, 3.5))
animator.add_caustic(label=r'Airy Caustic $x_c$')
animator.add_manifold(calc_x_manifold, x_sol_expr, [t_sym], color='cyan', prefix='Manifold')

t_values = np.linspace(-6.0, 6.0, 150)
animator.animate(t_values, compute_airy, lambda t: [t], lambda t: [[t]], 
                 lambda t: rf'Airy Transition ($t = {t:.2f}$) | $A_2$ Fold', '1d_airy_fold.gif')

### 2. True Pearcey Cusp Trajectory ($A_3$)

* **Mechanism:** Parameter trajectory tracking the cusp curve $y(\alpha) = \frac{\alpha^2}{3}$.
* **Phase Space:** Shows the cubic manifold inflection point turning vertical at $\xi_c = -\frac{\alpha}{3}$, representing the true $A_3$ Cusp Singularity.

In [ ]:
alpha_sym, y_sym = symbols('alpha y', real=True)
phi_pearcey = 0.25 * xi_sym**4 + (alpha_sym / 3.0) * xi_sym**3 + 0.5 * y_sym * xi_sym**2 + x_sym * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_pearcey, x_sym, xi_sym, [alpha_sym, y_sym])

xi_int = np.linspace(-3.5, 3.5, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.15 * xi_int**2)

def compute_pearcey(x_arr, alpha):
    y_cusp = (alpha**2) / 3.0
    phase = 0.25 * xi_int[:, None]**4 + (alpha / 3.0) * xi_int[:, None]**3 + 0.5 * y_cusp * xi_int[:, None]**2 + x_arr[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 1.5

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-3.5, 3.5))
animator.add_caustic(label=r'Cusp Position $x_c$')
animator.add_manifold(calc_x_manifold, x_sol_expr, [alpha_sym, y_sym], color='cyan', prefix='Cusp')

alpha_values = np.linspace(-3.0, 3.0, 150)
animator.animate(alpha_values, compute_pearcey, 
                 lambda a: [(a**3)/27.0], 
                 lambda a: [[a, (a**2)/3.0]], 
                 lambda a: rf'Cusp Trajectory ($\alpha = {a:.2f}$) | $A_3$', '1d_pearcey_cusp.gif')

### 3. Pearcey Unfolding / Dual Fold Coalescence ($A_3 \rightarrow A_2 \rightarrow A_1$)

* **Mechanism:** Sweeping $y$ through 0 at $\alpha = 0$.
* **Phase Space:** Visualizes two separate vertical fold tangents ($y < 0$) merging into a single cusp tangent ($y = 0$), then smoothing into a regular monotonic manifold ($y > 0$).

In [ ]:
y_sym = symbols('y', real=True)
phi_unfolding = 0.25 * xi_sym**4 + 0.5 * y_sym * xi_sym**2 + x_sym * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_unfolding, x_sym, xi_sym, [y_sym])

xi_int = np.linspace(-4.0, 4.0, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.12 * xi_int**2)

def compute_unfolding(x_arr, y_val):
    phase = 0.25 * xi_int[:, None]**4 + 0.5 * y_val * xi_int[:, None]**2 + x_arr[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 1.5

def get_caustics_unfolding(y):
    disc = -3.0 * y
    if disc > 0:
        xi_c = np.sqrt(disc / 3.0)
        return [-(xi_c**3 + y * xi_c), -((-xi_c)**3 + y * (-xi_c))]
    return [-999, -999]

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-3.5, 3.5))
animator.add_caustic(label='Caustic 1')
animator.add_caustic(label='Caustic 2')
animator.add_manifold(calc_x_manifold, x_sol_expr, [y_sym], color='cyan', prefix='Manifold')

y_values = np.linspace(-3.0, 1.5, 150)
animator.animate(y_values, compute_unfolding, get_caustics_unfolding, lambda y: [[y]], 
                 lambda y: rf'Unfolding Sweep ($y = {y:.2f}$)', '1d_pearcey_unfolding.gif')

### 4. Colliding Caustics (Interfering Airy Beams)

* **Mechanism:** Superposition of two oppositely propagating $A_2$ Airy fold caustics.
* **Phase Space:** Plots two intersecting parabolic manifolds $x_1(\xi)$ and $x_2(\xi)$ crossing each other in $(x, \xi)$ space, showing high-frequency standing-wave lattices at the phase-space collision point.

In [ ]:
xc1_sym, xc2_sym = symbols('x_{c1} x_{c2}', real=True)
phi_right = (1.0 / 3.0) * xi_sym**3 + (x_sym - xc1_sym) * xi_sym
phi_left = (1.0 / 3.0) * xi_sym**3 - (x_sym - xc2_sym) * xi_sym

calc_x_man1, x_sol1_expr, _ = create_manifold_evaluator(phi_right, x_sym, xi_sym, [xc1_sym])
calc_x_man2, x_sol2_expr, _ = create_manifold_evaluator(phi_left, x_sym, xi_sym, [xc2_sym])

xi_int = np.linspace(-4.0, 4.0, 501)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.1 * xi_int**2)

def compute_single_airy(x_shift):
    phase = (1.0 / 3.0) * xi_int[:, None]**3 + x_shift[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 2.0

def compute_colliding(x_arr, t_val):
    xc1, xc2 = -7.0 + 1.2 * t_val, 7.0 - 1.2 * t_val
    return compute_single_airy(x_arr - xc1) + compute_single_airy(-(x_arr - xc2))

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-4.5, 4.5))
animator.add_caustic(label=r'Caustic $x_{c1}$', color='red')
animator.add_caustic(label=r'Caustic $x_{c2}$', color='magenta')
animator.add_manifold(calc_x_man1, x_sol1_expr, [xc1_sym], color='cyan', prefix='Right')
animator.add_manifold(calc_x_man2, x_sol2_expr, [xc2_sym], color='yellow', prefix='Left')

t_values = np.linspace(0.0, 12.0, 180)
animator.animate(t_values, compute_colliding, 
                 lambda t: [-7.0 + 1.2*t, 7.0 - 1.2*t], 
                 lambda t: [[-7.0 + 1.2*t], [7.0 - 1.2*t]], 
                 lambda t: rf'Colliding Airy Beams ($t = {t:.2f}$)', '1d_colliding_airy.gif')

### 5. Swallowtail Singularity Trajectory ($A_4$)

* **Mechanism:** Parameter trajectory sweeping the control parameter $y$ across zero at a fixed $z < 0$ to isolate the swallowtail cross-section topology.
* **Phase Space:** Shows the quintic manifold transitioning from a 3-fold configuration (three vertical tangents) to a 1-fold configuration, passing through the degenerate $A_4$ swallowtail point at the inflection frequency $\xi_{sw} = \sqrt{-z/6}$.

In [ ]:
y_sym, z_sym = symbols('y z', real=True)
phi_swallowtail = (1.0 / 5.0) * xi_sym**5 + (z_sym / 3.0) * xi_sym**3 + (0.5 * y_sym) * xi_sym**2 + x_sym * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_swallowtail, x_sym, xi_sym, [y_sym, z_sym])

xi_int = np.linspace(-3.0, 3.0, 501)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.1 * xi_int**6)
z_fixed = -1.0
xi_sw = np.sqrt(-z_fixed / 6.0)

def compute_swallowtail(x_arr, y_val):
    phase = 0.2 * xi_int[:, None]**5 + (z_fixed / 3.0) * xi_int[:, None]**3 + 0.5 * y_val * xi_int[:, None]**2 + x_arr[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 1.5

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-3.5, 3.5), y_lim_spec=(-3.0, 3.0))
animator.add_caustic(label=r'Swallowtail $x_{sw}$')
animator.add_manifold(calc_x_manifold, x_sol_expr, [y_sym, z_sym], color='cyan', prefix='Swallowtail')

y_values = np.linspace(-1.5, 1.5, 150)
animator.animate(y_values, compute_swallowtail, 
                 lambda y: [-(xi_sw**4) - z_fixed * (xi_sw**2) - y * xi_sw], 
                 lambda y: [[y, z_fixed]], 
                 lambda y: rf'Swallowtail ($A_4$) | $y = {y:.2f}$', '1d_swallowtail.gif')

### 6. Capstone Trident Bifurcation & Barrier PsiDO

* **Mechanism:** Parameter sweep of $\mu$ through the quintic-cubic phase function $\phi(\xi; x, \mu) = \frac{1}{5}\xi^5 - \frac{\mu}{3}\xi^3 + x\xi$, coupled with a barrier-modulated fractional pseudo-differential operator $S(x, \xi)$.
* **Phase Space:** Plots the multi-fold trident-shaped Lagrangian manifold $x(\xi) = \mu\xi^2 - \xi^4$ transitioning through bifurcation points, displaying complex ray intersections and localized potential scattering lattices in $(x, \xi)$ space.

In [ ]:
mu_sym = symbols('mu', real=True)
phi_trident = (1.0 / 5.0) * xi_sym**5 - (mu_sym / 3.0) * xi_sym**3 + x_sym * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_trident, x_sym, xi_sym, [mu_sym])

S_capstone_expr = sqrt(1 + xi_sym**2) * (1 + 0.5 * exp(-x_sym**2)) + 0.1 * xi_sym * x_sym / (1 + x_sym**2)
OpCapstone = PseudoDifferentialOperator(expr=S_capstone_expr, vars_x=[x_sym], mode='symbol')

xi_int = np.linspace(-3.5, 3.5, 501)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.15 * xi_int**6)

def compute_trident(x_arr, mu_val):
    phase = 0.2 * xi_int[:, None]**5 - (mu_val / 3.0) * xi_int[:, None]**3 + x_arr[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 1.5

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, OpCapstone, y_lim_field=(-3.5, 3.5), y_lim_spec=(-3.2, 3.2), cmap='inferno')
animator.add_caustic(label=r'Bifurcation Apex')
animator.add_manifold(calc_x_manifold, x_sol_expr, [mu_sym], color='cyan', prefix='Trident')

mu_values = np.linspace(-2.5, 2.5, 150)
animator.animate(mu_values, compute_trident, lambda m: [0.0], lambda m: [[m]], 
                 lambda m: rf'Capstone Trident ($\mu = {m:.2f}$)', '1d_capstone_trident.gif')

### 7. Microlocal Branch Filtering (Isolating Interfering Caustic Rays)

* **Mechanism:** Application of a smooth momentum-space phase-space filter $S(\xi) = \frac{1}{2}\left[1 + \tanh\left(\frac{\xi - \xi_0}{\Delta \xi}\right)\right]$ to an Airy fold wavefield.
* **Phase Space:** Demonstrates selective surgical removal of one branch of the Lagrangian manifold $\mathcal{L}$, proving the power of $\psi\mathrm{Op}$ over standard spatial filters by resolving overlapping rays that share identical physical positions $x$ but distinct momenta $\xi$.

In [ ]:
t_sym = symbols('t', real=True)
phi_airy = (1.0 / 3.0) * xi_sym**3 + (x_sym - t_sym) * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_airy, x_sym, xi_sym, [t_sym])

S_filter_expr = 0.5 * (1 + tanh((xi_sym - 0.5) / 0.3))
OpFilter = PseudoDifferentialOperator(expr=S_filter_expr, vars_x=[x_sym], mode='symbol')

xi_int = np.linspace(-4.0, 4.0, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.1 * xi_int**2)

def compute_airy(x_arr, t_val):
    phase = (1.0 / 3.0) * xi_int[:, None]**3 + (x_arr[None, :] - t_val) * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 2.0

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, OpFilter, y_lim_field=(-2.5, 3.5))
animator.add_caustic(label=r'Caustic $x_c$')
animator.add_manifold(calc_x_manifold, x_sol_expr, [t_sym], color='cyan', prefix='Manifold')

t_values = np.linspace(-6.0, 6.0, 150)
animator.animate(t_values, compute_airy, lambda t: [t], lambda t: [[t]], 
                 lambda t: rf'Microlocal Filtering ($t = {t:.2f}$)', '1d_microlocal_filtering.gif')

### 8. Microlocal Frequency-Shift Modulation (Chirp Injection)

* **Mechanism:** Application of a position-momentum mixed symbol $S(x, \xi) = \xi + \omega_0 \operatorname{sech}^2(x)$ to an Airy fold wavefield.
* **Phase Space:** Dynamically tilts and shears the Lagrangian manifold $\mathcal{L}$ and spectrogram energy as the wave packet crosses a localized momentum-kick barrier, simulating nonlinear frequency conversion and microlocal lensing effects in phase space.

In [ ]:
t_sym = symbols('t', real=True)
phi_airy = (1.0 / 3.0) * xi_sym**3 + (x_sym - t_sym) * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_airy, x_sym, xi_sym, [t_sym])

S_chirp_expr = xi_sym + 1.5 * (1.0 / cosh(x_sym)**2)
OpChirp = PseudoDifferentialOperator(expr=S_chirp_expr, vars_x=[x_sym], mode='symbol')

xi_int = np.linspace(-4.0, 4.0, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.1 * xi_int**2)

def compute_airy(x_arr, t_val):
    phase = (1.0 / 3.0) * xi_int[:, None]**3 + (x_arr[None, :] - t_val) * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 2.0

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, OpChirp, y_lim_field=(-3.0, 3.5), y_lim_spec=(-4.5, 4.5), cmap='plasma')
animator.add_caustic(label=r'Caustic $x_c$')
animator.add_manifold(calc_x_manifold, x_sol_expr, [t_sym], color='cyan', prefix='Manifold')

t_values = np.linspace(-6.0, 6.0, 150)
animator.animate(t_values, compute_airy, lambda t: [t], lambda t: [[t]], 
                 lambda t: rf'Chirp Modulation ($t = {t:.2f}$)', '1d_microlocal_chirp.gif')

### 9. Pearcey Microlocal Branch Filtering (Inclined Phase-Space Knife-Edge)

* **Mechanism:** Application of an inclined phase-space knife-edge symbol $S_{\text{cusp}}(x, \xi) = \frac{1}{2}\left[1 + \tanh\left(\frac{\xi - 0.5 x}{\Delta}\right)\right]$ to a Pearcey cusp wavefield.
* **Phase Space:** Dynamically slices across the overlapping branches of the $A_3$ manifold, surgically isolating individual ray bundles and removing complex high-frequency interference diffraction patterns that share identical physical positions $x$.

In [ ]:
alpha_sym, y_sym = symbols('alpha y', real=True)
phi_pearcey = 0.25 * xi_sym**4 + (alpha_sym / 3.0) * xi_sym**3 + 0.5 * y_sym * xi_sym**2 + x_sym * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_pearcey, x_sym, xi_sym, [alpha_sym, y_sym])

S_cusp_filter = 0.5 * (1 + tanh((xi_sym - 0.5 * x_sym) / 0.4))
OpCuspFilter = PseudoDifferentialOperator(expr=S_cusp_filter, vars_x=[x_sym], mode='symbol')

xi_int = np.linspace(-3.5, 3.5, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.15 * xi_int**2)

def compute_pearcey(x_arr, alpha):
    y_cusp = (alpha**2) / 3.0
    phase = 0.25 * xi_int[:, None]**4 + (alpha / 3.0) * xi_int[:, None]**3 + 0.5 * y_cusp * xi_int[:, None]**2 + x_arr[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 1.5

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, OpCuspFilter, y_lim_field=(-3.5, 3.5))
animator.add_caustic(label=r'Cusp Position $x_c$')
animator.add_manifold(calc_x_manifold, x_sol_expr, [alpha_sym, y_sym], color='cyan', prefix='Cusp')

alpha_values = np.linspace(-3.0, 3.0, 150)
animator.animate(alpha_values, compute_pearcey, 
                 lambda a: [(a**3)/27.0], 
                 lambda a: [[a, (a**2)/3.0]], 
                 lambda a: rf'Pearcey Filtering ($\alpha = {a:.2f}$)', '1d_pearcey_filter.gif')

### 10. Colliding Airy-Pearcey Wavefield Catastrophes

* **Mechanism:** Superposition and mutual crossing of an Airy fold beam ($A_2$ catastrophe) and a dynamic Pearcey cusp beam ($A_3$ catastrophe) traveling in opposite directions.
* **Phase Space:** Simultaneously overlays single-branch fold manifolds and multi-branched folding cusp structures, tracking hybrid catastrophe interactions and high-order diffraction lattice interference in time-frequency space.

In [ ]:
xc1_sym = symbols('x_{c1}', real=True)
alpha_sym, y_sym, xc2_sym = symbols('alpha y x_{c2}', real=True)

phi_airy = (1.0 / 3.0) * xi_sym**3 + (x_sym - xc1_sym) * xi_sym
phi_pearcey = 0.25 * xi_sym**4 + (alpha_sym / 3.0) * xi_sym**3 + 0.5 * y_sym * xi_sym**2 + (x_sym - xc2_sym) * xi_sym

calc_x_man1, x_sol1_expr, _ = create_manifold_evaluator(phi_airy, x_sym, xi_sym, [xc1_sym])
calc_x_man2, x_sol2_expr, _ = create_manifold_evaluator(phi_pearcey, x_sym, xi_sym, [alpha_sym, y_sym, xc2_sym])

xi_int = np.linspace(-3.5, 3.5, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.15 * xi_int**2)

def compute_single_airy(x_shift):
    phase = (1.0 / 3.0) * xi_int[:, None]**3 + x_shift[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 2.0

def compute_pearcey_field(x_arr, alpha, y_param, x_shift):
    phase = 0.25 * xi_int[:, None]**4 + (alpha / 3.0) * xi_int[:, None]**3 + 0.5 * y_param * xi_int[:, None]**2 + (x_arr[None, :] - x_shift) * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 1.5

def compute_colliding_ap(x_arr, t_val):
    xc1, xc2 = -7.0 + 1.0 * t_val, 7.0 - 1.0 * t_val
    alpha_val = 1.5 * np.sin(0.2 * t_val)
    y_val = (alpha_val**2) / 3.0
    return compute_single_airy(x_arr - xc1) + compute_pearcey_field(x_arr, alpha_val, y_val, xc2)

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-4.5, 4.5))
animator.add_caustic(label=r'Airy $x_{c1}$', color='red')
animator.add_caustic(label=r'Pearcey $x_{c2}$', color='magenta')
animator.add_manifold(calc_x_man1, x_sol1_expr, [xc1_sym], color='cyan', prefix='Airy')
animator.add_manifold(calc_x_man2, x_sol2_expr, [alpha_sym, y_sym, xc2_sym], color='yellow', prefix='Pearcey')

t_values = np.linspace(0.0, 14.0, 180)
def get_params(t):
    xc1, xc2 = -7.0 + 1.0 * t, 7.0 - 1.0 * t
    alpha_val = 1.5 * np.sin(0.2 * t)
    return [[xc1], [alpha_val, (alpha_val**2)/3.0, xc2]]

animator.animate(t_values, compute_colliding_ap, 
                 lambda t: [-7.0 + 1.0*t, 7.0 - 1.0*t], get_params, 
                 lambda t: rf'Airy-Pearcey Collision ($t = {t:.2f}$)', '1d_colliding_ap.gif')

### 11. Filtered Airy-Pearcey Collision

* **Mechanism:** Application of a targeted momentum-space filter symbol $S_{\text{target}}(x, \xi) = \frac{1}{2}\left[1 + \tanh\left(\frac{\xi - 0.2}{0.3}\right)\right]$ to a superposed Airy-Pearcey colliding wavefield.
* **Phase Space:** Selectively isolates specific momentum branches and strips away competing high-order diffraction lattice interference patterns during the complex nonlinear crossing of $A_2$ fold and $A_3$ cusp catastrophe manifolds.

In [ ]:
S_target_expr = 0.5 * (1 + tanh((xi_sym - 0.2) / 0.3))
OpFilteredCollision = PseudoDifferentialOperator(expr=S_target_expr, vars_x=[x_sym], mode='symbol')

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, OpFilteredCollision, y_lim_field=(-4.5, 4.5))
animator.add_caustic(label=r'Airy $x_{c1}$', color='red')
animator.add_caustic(label=r'Pearcey $x_{c2}$', color='magenta')
animator.add_manifold(calc_x_man1, x_sol1_expr, [xc1_sym], color='cyan', prefix='Airy')
animator.add_manifold(calc_x_man2, x_sol2_expr, [alpha_sym, y_sym, xc2_sym], color='yellow', prefix='Pearcey')

animator.animate(t_values, compute_colliding_ap, 
                 lambda t: [-7.0 + 1.0*t, 7.0 - 1.0*t], get_params, 
                 lambda t: rf'Filtered AP Collision ($t = {t:.2f}$)', '1d_filtered_ap.gif')

### 12. Vertical Caustic Sweep (Frequency-Axis Translation)

* **Mechanism:** Modulation of the Airy phase function via a time-varying frequency center $\xi_0(t)$, modifying the stationary phase relation to $\phi(\xi; x, t) = \frac{1}{3}(\xi - \xi_0(t))^3 + x\xi$.
* **Phase Space:** Dynamically translates the entire parabolic Lagrangian manifold and its vertical turning point (caustic vertex) up and down along the local frequency axis $\xi$ in the time-frequency spectrogram.

In [ ]:
xi0_sym = symbols('xi_0', real=True)
phi_vertical = (1.0 / 3.0) * (xi_sym - xi0_sym)**3 + x_sym * xi_sym
calc_x_manifold, x_sol_expr, _ = create_manifold_evaluator(phi_vertical, x_sym, xi_sym, [xi0_sym])

xi_int = np.linspace(-4.0, 4.0, 401)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.1 * xi_int**2)

def compute_vertical(x_arr, xi0_val):
    phase = (1.0 / 3.0) * (xi_int[:, None] - xi0_val)**3 + x_arr[None, :] * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 2.0

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-3.0, 3.5), y_lim_spec=(-4.5, 4.5), cmap='plasma')
animator.add_caustic(label=r'Caustic $x_c = 0$')
animator.add_manifold(calc_x_manifold, x_sol_expr, [xi0_sym], color='cyan', prefix='Manifold')

t_values = np.linspace(0.0, 10.0, 150)
animator.animate(t_values, lambda x, t: compute_vertical(x, 1.5 * np.sin(0.5 * t)), 
                 lambda t: [0.0], lambda t: [[1.5 * np.sin(0.5 * t)]], 
                 lambda t: rf'Vertical Sweep ($t = {t:.2f}$)', '1d_vertical_sweep.gif')

### 13. Mutually Interacting Airy-Pearcey Collisions (Trajectory Deflection)

* **Mechanism:** Implementation of a soft repulsive potential barrier and nonlinear parameter feedback that dynamically shifts the caustic positions ($x_{c1}, x_{c2}$) and expands the Pearcey cusp parameter ($\alpha(t)$) near the collision point.
* **Phase Space:** Models mutual feedback where the Lagrangian manifolds deflect and push each other apart upon approach, dynamically deforming their spatial trajectories and caustic structures rather than passing through each other linearly.


In [ ]:
def compute_interacting(x_arr, t_val):
    t_col = 7.0
    base_xc1, base_xc2 = -7.0 + 1.0 * t_val, 7.0 - 1.0 * t_val
    repulsion = 2.2 / (1.0 + np.cosh(0.8 * (t_val - t_col)))
    xc1, xc2 = base_xc1 - repulsion, base_xc2 + repulsion
    
    alpha_val = 1.5 + 1.2 * np.exp(-0.5 * (t_val - t_col)**2 / 3.0)
    y_val = (alpha_val**2) / 3.0
    
    return compute_single_airy(x_arr - xc1) + compute_pearcey_field(x_arr, alpha_val, y_val, xc2)

def get_interacting_params(t):
    t_col = 7.0
    repulsion = 2.2 / (1.0 + np.cosh(0.8 * (t - t_col)))
    xc1, xc2 = -7.0 + 1.0 * t - repulsion, 7.0 - 1.0 * t + repulsion
    alpha_val = 1.5 + 1.2 * np.exp(-0.5 * (t - t_col)**2 / 3.0)
    return [[xc1], [alpha_val, (alpha_val**2)/3.0, xc2]]

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-4.5, 4.5))
animator.add_caustic(label=r'Airy $x_{c1}$', color='red')
animator.add_caustic(label=r'Pearcey $x_{c2}$', color='magenta')
animator.add_manifold(calc_x_man1, x_sol1_expr, [xc1_sym], color='cyan', prefix='Airy')
animator.add_manifold(calc_x_man2, x_sol2_expr, [alpha_sym, y_sym, xc2_sym], color='yellow', prefix='Pearcey')

t_values = np.linspace(0.0, 14.0, 180)
animator.animate(t_values, compute_interacting, 
                 lambda t: get_interacting_params(t)[0], 
                 get_interacting_params, 
                 lambda t: rf'Interacting Collision ($t = {t:.2f}$)', '1d_interacting.gif')

### 14. Phase-Space Rotation & Interaction (Symplectic Shearing)

* **Mechanism:** Integration of a time-dependent phase-space rotation symbol $S_{\text{rot}}(x, \xi) = \xi \cos\theta - x \sin\theta$ evaluated dynamically via numerical parameter injection, coupled with cross-coupled phase modulations in the wavefield computation.
* **Phase Space:** Dynamically rotates, twists, and shears the catastrophe manifolds and spectrogram energy distributions, providing a clear visual demonstration of canonical phase-space transformations and shearing during wavefield scattering.

In [ ]:
theta_sym = symbols('theta', real=True)
phi_airy_rot = (1.0 / 3.0) * xi_sym**3 + (x_sym - xc1_sym) * xi_sym + 0.3 * theta_sym * x_sym * xi_sym
phi_pearcey_rot = 0.25 * xi_sym**4 + (alpha_sym / 3.0) * xi_sym**3 + 0.5 * y_sym * xi_sym**2 + (x_sym - xc2_sym) * xi_sym - 0.2 * theta_sym * x_sym * xi_sym

calc_x_man1_rot, x_sol1_rot, _ = create_manifold_evaluator(phi_airy_rot, x_sym, xi_sym, [xc1_sym, theta_sym])
calc_x_man2_rot, x_sol2_rot, _ = create_manifold_evaluator(phi_pearcey_rot, x_sym, xi_sym, [alpha_sym, y_sym, xc2_sym, theta_sym])

def compute_rotated(x_arr, t_val):
    t_col = 7.0
    repulsion = 2.5 / (1.0 + np.cosh(0.8 * (t_val - t_col)))
    xc1, xc2 = -7.0 + 1.0 * t_val - repulsion, 7.0 - 1.0 * t_val + repulsion
    theta_val = 0.6 * np.tanh(0.6 * (t_val - t_col))
    
    # 1. Define numerical values directly to avoid SymPy/NumPy mixing issues
    alpha_num = 1.5
    y_num = 0.75
    
    # 2. Construct phases using pure NumPy operations
    phase1 = (1.0 / 3.0) * xi_int[:, None]**3 + (x_arr[None, :] - xc1) * xi_int[:, None] + 0.3 * theta_val * (x_arr[None, :] - xc1) * xi_int[:, None]
    phase2 = 0.25 * xi_int[:, None]**4 + (alpha_num / 3.0) * xi_int[:, None]**3 + 0.5 * y_num * xi_int[:, None]**2 + (x_arr[None, :] - xc2) * xi_int[:, None] - 0.2 * theta_val * (x_arr[None, :] - xc2) * xi_int[:, None]
    
    # 3. Evaluate wavefields
    u1 = np.sum(np.exp(1j * phase1) * xi_damping[:, None], axis=0) * dxi / 2.0
    u2 = np.sum(np.exp(1j * phase2) * xi_damping[:, None], axis=0) * dxi / 1.5
    u_raw = u1 + u2
    
    # 4. Apply dynamic operator
    S_rot_expr = xi_sym * np.cos(theta_val) - x_sym * np.sin(theta_val)
    OpRot = PseudoDifferentialOperator(expr=S_rot_expr, vars_x=[x_sym], mode='symbol')
    Pu = OpRot.apply(u_raw * window, x_grid, kx, clamp=1e6, space_window=False)
    
    return u_raw, Pu

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-4.5, 4.5), y_lim_spec=(-4.5, 4.5))
animator.add_caustic(label=r'Airy $x_{c1}$', color='red')
animator.add_caustic(label=r'Pearcey $x_{c2}$', color='magenta')
animator.add_manifold(calc_x_man1_rot, x_sol1_rot, [xc1_sym, theta_sym], color='cyan', prefix='Airy')
animator.add_manifold(calc_x_man2_rot, x_sol2_rot, [alpha_sym, y_sym, xc2_sym, theta_sym], color='yellow', prefix='Pearcey')

def get_rot_params(t):
    t_col = 7.0
    repulsion = 2.5 / (1.0 + np.cosh(0.8 * (t - t_col)))
    xc1, xc2 = -7.0 + 1.0 * t - repulsion, 7.0 - 1.0 * t + repulsion
    theta_val = 0.6 * np.tanh(0.6 * (t - t_col))
    alpha_val = 1.5 + 1.5 * np.exp(-0.5 * (t - t_col)**2 / 2.5)
    return [[xc1, theta_val], [alpha_val, (alpha_val**2)/3.0, xc2, theta_val]]

animator.animate(t_values, compute_rotated, 
                 lambda t: [get_rot_params(t)[0][0], get_rot_params(t)[1][2]], 
                 get_rot_params, 
                 lambda t: rf'Phase-Space Rotation ($t = {t:.2f}$)', '1d_rotation.gif')

### 15. Topological Manifold Warping & Structural Transformation

* **Mechanism:** Implementation of a nonlinearly coupled phase function featuring an algebraic polynomial interaction term ($\frac{1}{2} \gamma(t) x \xi^3$). This ensures the stationary phase equation remains analytically solvable by SymPy while introducing active spatial-frequency feedback.
* **Phase Space:** Dynamically bends, stretches, and restructures the internal topology of the Lagrangian manifold branches. Rather than acting as rigid shapes or undergoing simple linear translation, the caustics and cusp structures physically warp and deform as the interaction parameter peaks.


In [ ]:
xc_sym, gamma_sym = symbols('x_c gamma', real=True)
phi_interacting = 0.25 * xi_sym**4 + (1.5 + 0.5 * gamma_sym * x_sym) * xi_sym**3 + 0.5 * xi_sym**2 + (x_sym - xc_sym) * xi_sym
calc_x_man, x_sol_expr, _ = create_manifold_evaluator(phi_interacting, x_sym, xi_sym, [xc_sym, gamma_sym])

xi_int = np.linspace(-4.5, 4.5, 501)
dxi = xi_int[1] - xi_int[0]
xi_damping = np.exp(-0.12 * xi_int**2)

def compute_warped(x_arr, xc_val, gamma_val):
    phase = 0.25 * xi_int[:, None]**4 + (1.5 + 0.5 * gamma_val * x_arr[None, :]) * xi_int[:, None]**3 + 0.5 * xi_int[:, None]**2 + (x_arr[None, :] - xc_val) * xi_int[:, None]
    return np.sum(np.exp(1j * phase) * xi_damping[:, None], axis=0) * dxi / 2.0

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-5.0, 5.0), y_lim_spec=(-4.5, 4.5))
animator.add_caustic(label=r'Dynamic Center $x_c$')
animator.add_manifold(calc_x_man, x_sol_expr, [xc_sym, gamma_sym], color='cyan', prefix='Warped')

t_values = np.linspace(0.0, 14.0, 180)
def get_warp_params(t):
    xc_val = -7.0 + 1.0 * t
    gamma_val = 2.5 / (1.0 + np.cosh(0.8 * (t - 7.0)))
    return [[xc_val, gamma_val]]

animator.animate(t_values, lambda x, t: compute_warped(x, -7.0 + 1.0*t, 2.5 / (1.0 + np.cosh(0.8 * (t - 7.0)))), 
                 lambda t: [get_warp_params(t)[0][0]], get_warp_params, 
                 lambda t: rf'Topological Warping ($t = {t:.2f}$)', '1d_warping.gif')

### 16. Dual Interacting Warped Lagrangian Manifolds & Topological Deformation

* **Mathematical Architecture & Solvability:** Built upon two distinct phase functions ($\phi_1$ and $\phi_2$) incorporating polynomial cross-coupling terms ($\gamma(t) x \xi^3$). By keeping the spatial coordinate $x$ strictly linear within these coupling terms, the stationary phase condition ($\partial_\xi \phi = 0$) remains algebraically invertible. This allows SymPy to solve for explicit rational functions $x_1(\xi)$ and $x_2(\xi)$ without encountering transcendental solver limitations.
* **Kinematic Interaction & Trajectory Evolution:** The two wave systems feature time-dependent centers ($x_{c1}(t)$ and $x_{c2}(t)$) that converge and experience a soft repulsive barrier governed by a hyperbolic secant potential near the collision time ($t_{\text{col}} = 7.0$).
* **Topological Morphing:** As the mutual interaction strength parameter $\gamma(t)$ peaks during the approach, it dynamically alters the denominators and coefficients of both manifold equations. Consequently, the cyan and yellow analytical curves do not merely translate as rigid shapes; their internal branches, folds, and caustic structures physically warp, stretch, and restructure in real-time.
* **Spectrogram Synchronization:** The Short-Time Fourier Transform (STFT) spectrogram captures the high-intensity energy distributions of the dual wavefields. The dynamically evaluated analytical curves overlay these energy patches precisely, illustrating how phase-space geometry deforms concurrently with the wave packet collisions.


In [ ]:
xc1_sym, xc2_sym, gamma_sym = symbols('x_{c1} x_{c2} gamma', real=True)
phi_1 = (1.0 / 3.0) * xi_sym**3 + (1.0 + 0.4 * gamma_sym * x_sym) * xi_sym**3 + (x_sym - xc1_sym) * xi_sym
phi_2 = 0.25 * xi_sym**4 + (1.5 + 0.4 * gamma_sym * x_sym) * xi_sym**3 + 0.5 * xi_sym**2 + (x_sym - xc2_sym) * xi_sym

calc_x_man1, x_sol1_expr, _ = create_manifold_evaluator(phi_1, x_sym, xi_sym, [xc1_sym, gamma_sym])
calc_x_man2, x_sol2_expr, _ = create_manifold_evaluator(phi_2, x_sym, xi_sym, [xc2_sym, gamma_sym])

def compute_dual_warped(x_arr, xc1_val, xc2_val, gamma_val):
    phase1 = (1.0 / 3.0) * xi_int[:, None]**3 + (1.0 + 0.4 * gamma_val * x_arr[None, :]) * xi_int[:, None]**3 + (x_arr[None, :] - xc1_val) * xi_int[:, None]
    phase2 = 0.25 * xi_int[:, None]**4 + (1.5 + 0.4 * gamma_val * x_arr[None, :]) * xi_int[:, None]**3 + 0.5 * xi_int[:, None]**2 + (x_arr[None, :] - xc2_val) * xi_int[:, None]
    u1 = np.sum(np.exp(1j * phase1) * xi_damping[:, None], axis=0) * dxi / 2.0
    u2 = np.sum(np.exp(1j * phase2) * xi_damping[:, None], axis=0) * dxi / 1.5
    return u1 + u2

animator = PhaseSpaceAnimator(x_grid, kx, dx, L, window, SFT, Op1D, y_lim_field=(-5.0, 5.0), y_lim_spec=(-4.5, 4.5))
animator.add_caustic(label=r'Center $x_{c1}$', color='red')
animator.add_caustic(label=r'Center $x_{c2}$', color='magenta')
animator.add_manifold(calc_x_man1, x_sol1_expr, [xc1_sym, gamma_sym], color='cyan', prefix='Manifold 1')
animator.add_manifold(calc_x_man2, x_sol2_expr, [xc2_sym, gamma_sym], color='yellow', prefix='Manifold 2')

def get_dual_params(t):
    repulsion = 2.0 / (1.0 + np.cosh(0.8 * (t - 7.0)))
    xc1, xc2 = -7.0 + 1.0 * t - repulsion, 7.0 - 1.0 * t + repulsion
    gamma_val = 2.5 / (1.0 + np.cosh(0.8 * (t - 7.0)))
    return [[xc1, gamma_val], [xc2, gamma_val]]

animator.animate(t_values, lambda x, t: compute_dual_warped(x, *get_dual_params(t)[0], *get_dual_params(t)[1][1:]), 
                 lambda t: [get_dual_params(t)[0][0], get_dual_params(t)[1][0]], 
                 get_dual_params, 
                 lambda t: rf'Dual Warped Manifolds ($t = {t:.2f}$)', '1d_dual_warped.gif')

### 17. True Nonlinear Wave Mechanics & Dynamic Manifold Extraction (NLSE)

* **Mechanism:** Integration of the 1D Non-Linear Schrödinger Equation (NLSE) $i u_t + u_{xx} + \chi \vert{}u\vert{}^2 u = 0$ via a pseudospectral solver. Two counter-propagating Gaussian wavepackets undergo collision under Kerr non-linearity ($\chi = 2.5$), driving self-phase modulation and active non-linear phase coupling.
* **Phase Space:** Real-time tracking of the evolving 1D Lagrangian manifold $\xi(x,t) = \operatorname{Im}\left(\frac{\partial_x u}{u}\right)$ via Madelung hydrodynamic phase derivatives and high-accuracy spectral gradients. Overlaid on a centered Short-Time Fourier Transform (STFT) spectrogram, it directly resolves the non-linear deformation, phase-space self-focusing, and temporal interference structures resulting from wavepacket interaction.

In [ ]:
# ==========================================
# 1. NONLINEAR PDE DEFINITION (NLSE)
# ==========================================
# Equation: i * u_t + u_xx + chi * |u|^2 * u = 0
# Rewritten for solver: u_t = i * u_xx + i * chi * |u|^2 * u
t, x = symbols('t x')
u = Function('u')(t, x)
chi = 2.5  # Kerr non-linearity strength

# SymPy equation definition
eq = Eq(
    diff(u, t), 
    1j * diff(u, x, 2) + 1j * chi * Abs(u)**2 * u
)

# Initialize your PDESolver
solver = PDESolver(eq, time_scheme='ETD-RK4', dealiasing_ratio=2/3, compute_energy=False)

# Setup Grid and Initial Conditions (Two counter-propagating Gaussian wavepackets)
Lx, Nx = 60.0, 2048
Lt, Nt = 15.0, 3000 

x_grid = np.linspace(-Lx/2, Lx/2, Nx, endpoint=False)
dx = x_grid[1] - x_grid[0]  # Compute dx manually
k0 = 2.0  # Initial momentum separation
x1_0, x2_0 = -15.0, 15.0
w0 = 2.5

def initial_condition(x_arr):
    u1 = np.exp(-(x_arr - x1_0)**2 / w0**2 + 1j * k0 * x_arr)
    u2 = np.exp(-(x_arr - x2_0)**2 / w0**2 - 1j * k0 * x_arr)
    return u1 + u2

solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition, plot=False)

# ==========================================
# 2. VISUALIZATION & ANIMATION
# ==========================================
# FIX: Use the locally computed dx instead of solver.dx
SFT = ShortTimeFFT(gaussian(128, std=16), hop=4, fs=1/dx, scale_to='magnitude', fft_mode='centered')

fig, (ax_field, ax_spec) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={'height_ratios': [1, 1.2]})

line_u, = ax_field.plot([], [], color="tab:blue", lw=1.5, label=r"Nonlinear Wavefield $u(x,t)$")
ax_field.set_ylabel("Amplitude")
ax_field.set_ylim(-2.5, 2.5)
ax_field.grid(True)
ax_field.legend(loc="upper left", fontsize=8)

im_spec = ax_spec.imshow(np.zeros((SFT.f_pts, SFT.p_num(Nx))), aspect='auto', origin='lower', cmap='magma')
ax_spec.set_xlabel("Spatial Coordinate $x$")
ax_spec.set_ylabel(r"Instantaneous Frequency $\xi(x,t)$")
ax_spec.set_ylim(-6.0, 6.0)

# Scatter plot for the dynamically extracted manifold
scat_man = ax_spec.scatter([], [], color="cyan", s=1.5, alpha=0.8, label=r"Instantaneous Manifold $\partial_x \arg(u)$")
ax_spec.legend(loc="upper left", fontsize=8)

# ==========================================
# 3. ANIMATION LOOP (Coupling Solver to Phase-Space)
# ==========================================
# We will step the solver manually frame-by-frame to sync with Matplotlib
frames_to_save = 150
steps_per_frame = Nt // frames_to_save

current_step = 0
u_current = solver.u_prev.copy()

def update_nlse(frame_idx):
    global current_step, u_current
    
    # Evolve the PDE using your ETD-RK4 engine
    for _ in range(steps_per_frame):
        # Using the internal ETD-RK4 step for 1st order complex PDEs
        u_current = solver._step_ETD_RK4_order1(u_current, t=current_step * solver.dt)
        current_step += 1
        
    t_current = current_step * solver.dt
    
    # 1. Update Physical Field
    line_u.set_data(x_grid, np.real(u_current))
    
    # 2. Update Spectrogram
    spec_mag = np.abs(SFT.stft(u_current))
    t_stft = SFT.t(Nx) - Lx/2
    xi_stft = 2 * np.pi * SFT.f
    im_spec.set_data(spec_mag)
    im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
    im_spec.set_clim(vmin=0, vmax=spec_mag.max() * 0.8)
    
    # 3. Extract & Plot the Nonlinear Manifold
    xi_inst, mask = compute_instantaneous_manifold(u_current, dx, method='madelung', threshold=0.05)
    
    # Downsample for scatter plot performance
    step = 4
    scat_man.set_offsets(np.c_[x_grid[::step][mask[::step]], xi_inst[::step][mask[::step]]])
    
    ax_field.set_title(rf"True Nonlinear Interaction (NLSE) | $t = {t_current:.2f}$ | Kerr Effect $\chi = {chi}$")
    
    return line_u, im_spec, scat_man

# Generate Animation
t_frames = np.linspace(0, 12.0, 150)
anim_nlse = FuncAnimation(fig, update_nlse, frames=len(t_frames), interval=60)
anim_nlse.save("17_true_nlse_manifold_interaction.gif", writer="pillow", fps=15)
plt.show()

### 18. Dispersive Shock Waves & Hilbert Phase-Space Manifold (KdV)

* **Mechanism:** Numerical integration of the Korteweg–de Vries (KdV) equation $u_t + u u_x + u_{xxx} = 0$ initialized with a high-amplitude Gaussian pulse using an ETD-RK4 pseudospectral solver. Convective non-linear steepening induces a gradient catastrophe, which is regularized by third-order dispersion into a rank-ordered soliton train and dispersive shock wave (DSW).
* **Phase Space:** Real-time reconstruction of the instantaneous spatial wavenumber $\xi(x,t) = \partial_x \arg(u + \mathrm{i}\mathcal{H}[u])$ for real-valued fields using the complex analytic signal generated via the Hilbert transform ($\mathcal{H}$). Superimposed over a Short-Time Fourier Transform (STFT) spectrogram, the extracted 1D Lagrangian manifold directly captures the spectral splitting, localized wave-number jumps, and phase-space lattice dynamics of the emerging multi-soliton structures.

In [ ]:
# ==========================================
# 1. DEFINE THE KDV EQUATION
# ==========================================
# Equation: u_t + u*u_x + u_xxx = 0
# Rearranged for solver: u_t = -u*u_x - u_xxx
t, x = symbols('t x')
u = Function('u')(t, x)

eq = Eq(
    diff(u, t), 
    -u * diff(u, x) - diff(u, x, 3)
)

# Initialize the ETD-RK4 Spectral Solver
solver = PDESolver(eq, time_scheme='ETD-RK4', dealiasing_ratio=2/3, compute_energy=False)

# Setup Grid and Initial Conditions
Lx, Nx = 100.0, 2048
Lt, Nt = 15.0, 6000  # High resolution to capture the soliton train

# Initial condition: A broad, high-amplitude Gaussian pulse
def initial_condition(x_arr):
    return 5.0 * np.exp(-x_arr**2 / 8.0)

solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition, plot=False)

# ==========================================
# 2. SOLVE THE PDE (Pre-computation)
# ==========================================
print("\nSolving KdV equation...")
solver.solve()
print("Solving complete. Preparing animation...")


# ==========================================
# 3. STFT & VISUALIZATION SETUP
# ==========================================
dx = Lx / Nx
SFT = ShortTimeFFT(gaussian(256, std=32), hop=8, fs=1/dx, scale_to='magnitude', fft_mode='centered')

fig, (ax_field, ax_spec) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.5]})

line_u, = ax_field.plot([], [], color="tab:blue", lw=1.5, label=r"KdV Wavefield $u(x,t)$")
ax_field.set_ylabel("Amplitude $u$")
ax_field.set_ylim(-1.0, 6.5)
ax_field.grid(True)
ax_field.legend(loc="upper left", fontsize=10)

im_spec = ax_spec.imshow(np.zeros((SFT.f_pts, SFT.p_num(Nx))), aspect='auto', origin='lower', cmap='magma')
ax_spec.set_xlabel("Spatial Coordinate $x$")
ax_spec.set_ylabel(r"Instantaneous Wavenumber $\xi(x,t)$")
ax_spec.set_ylim(-15.0, 15.0)

# Scatter plot for the dynamically extracted Hilbert manifold
scat_man = ax_spec.scatter([], [], color="cyan", s=2.0, alpha=0.9, label=r"Manifold $\partial_x \arg(u + i\mathcal{H}[u])$")
ax_spec.legend(loc="upper left", fontsize=10)

# ==========================================
# 4. ANIMATION LOOP
# ==========================================
# Map solver frames to animation frames
frames_to_animate = len(solver.frames)
step = max(1, frames_to_animate // 200) # Limit animation to ~200 frames for smooth GIF

def update_kdv(frame_idx):
    u_current = solver.frames[frame_idx * step]
    t_current = frame_idx * step * solver.dt
    
    # 1. Update Physical Field
    line_u.set_data(solver.x_grid, np.real(u_current))
    
    # 2. Update Spectrogram
    spec_mag = np.abs(SFT.stft(u_current))
    t_stft = SFT.t(Nx) - Lx/2
    xi_stft = 2 * np.pi * SFT.f
    im_spec.set_data(spec_mag)
    im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
    im_spec.set_clim(vmin=0, vmax=spec_mag.max() * 0.6)
    
    # 3. Extract & Plot the Nonlinear Manifold via Hilbert
    # Inside the update_kdv(frame_idx) function:
    xi_inst, mask = compute_instantaneous_manifold(u_current, dx, method='hilbert', threshold=0.05)
    
    # Downsample for scatter plot performance
    ds_step = 4
    scat_man.set_offsets(np.c_[solver.x_grid[::ds_step][mask[::ds_step]], xi_inst[::ds_step][mask[::ds_step]]])
    
    ax_field.set_title(rf"KdV Gradient Catastrophe & Dispersive Shock Wave | $t = {t_current:.2f}$")
    
    return line_u, im_spec, scat_man

anim_kdv = FuncAnimation(fig, update_kdv, frames=len(solver.frames) // step, interval=60)
anim_kdv.save("18_kdv_gradient_catastrophe.gif", writer="pillow", fps=15)
plt.show()

### 19. Nonlinear Manifold Collision & Hydrodynamic Phase-Space Dynamics (NLSE)

* **Mechanism:** Pseudospectral integration of the Non-Linear Schrödinger Equation (NLSE) $i u_t + u_{xx} + \chi \vert{}u\vert{}^2 u = 0$ using a PDE solver with enhanced Kerr non-linearity ($\chi = 3.0$). Two counter-propagating Gaussian wavepackets initialized with opposite momentum channels ($k_0 = \pm 2.5$) undergo direct collision, inducing strong local self-phase modulation and wavefield distortion.
* **Phase Space:** Dynamic extraction of the instantaneous 1D Lagrangian manifold $\xi(x,t) = \operatorname{Im}\left(\frac{\partial_x u}{u}\right)$ using the Madelung hydrodynamic identity implemented via spectral differentiation. Superimposed on a Short-Time Fourier Transform (STFT) spectrogram, the framework tracks the non-linear folding, momentum channel crossover, and transient phase singularities emerging during wavepacket interaction.

In [ ]:
# ==========================================
# 1. DEFINE THE NONLINEAR SCHRÖDINGER EQUATION (NLSE)
# ==========================================
# Equation: i * u_t + u_xx + chi * |u|^2 * u = 0
# Rearranged for solver: u_t = i * u_xx + i * chi * |u|^2 * u
t, x = symbols('t x', real=True)
u = Function('u')(t, x)
chi = 3.0  # Kerr non-linearity strength (increase for stronger repulsion)

# We use u * Abs(u)**2 to ensure the parser correctly identifies it as a nonlinear term.
# SymPy will parse this, and NumPy will evaluate it as field * np.abs(field)**2.
eq = Eq(
    diff(u, t), 
    1j * diff(u, x, 2) + 1j * chi * u * Abs(u)**2
)

# Initialize the ETD-RK4 Spectral Solver
# ETD-RK4 handles the stiff linear dispersion (u_xx) exactly in Fourier space,
# preventing numerical phase errors from smearing the caustic structures.
solver = PDESolver(eq, time_scheme='ETD-RK4', dealiasing_ratio=2/3, compute_energy=False)

# ==========================================
# 2. SETUP GRID & INITIAL CONDITIONS (TWO COLLIDING WAVES)
# ==========================================
Lx, Nx = 80.0, 2048
Lt, Nt = 15.0, 4000  # High resolution to capture sharp phase gradients during collision

# Initial condition: Two Gaussian wavepackets with opposite momenta
w0 = 3.0       # Beam width
k0 = 2.5       # Initial momentum separation (creates distinct manifolds in phase space)
x1_0, x2_0 = -20.0, 20.0

def initial_condition(x_arr):
    # Wave 1: Moving right (momentum +k0)
    u1 = np.exp(-(x_arr - x1_0)**2 / w0**2 + 1j * k0 * x_arr)
    # Wave 2: Moving left (momentum -k0)
    u2 = np.exp(-(x_arr - x2_0)**2 / w0**2 - 1j * k0 * x_arr)
    return u1 + u2

solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition, plot=False)

# ==========================================
# 3. SOLVE THE PDE (Pre-computation)
# ==========================================
print("\nSolving NLSE with colliding wavepackets...")
solver.solve()
print("Solving complete. Preparing animation...")


# ==========================================
# 4. STFT & VISUALIZATION SETUP
# ==========================================
dx = Lx / Nx
# Use a slightly wider window for better frequency resolution of the manifolds
SFT = ShortTimeFFT(gaussian(256, std=32), hop=8, fs=1/dx, scale_to='magnitude', fft_mode='centered')

fig, (ax_field, ax_spec) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.5]})

line_u, = ax_field.plot([], [], color="tab:blue", lw=1.5, label=r"Nonlinear Wavefield $\mathrm{Re}(u)$")
ax_field.set_ylabel("Amplitude")
ax_field.set_ylim(-2.5, 2.5)
ax_field.grid(True)
ax_field.legend(loc="upper left", fontsize=10)

im_spec = ax_spec.imshow(np.zeros((SFT.f_pts, SFT.p_num(Nx))), aspect='auto', origin='lower', cmap='magma')
ax_spec.set_xlabel("Spatial Coordinate $x$")
ax_spec.set_ylabel(r"Instantaneous Frequency $\xi(x,t)$")
ax_spec.set_ylim(-6.0, 6.0)

# Scatter plot for the dynamically extracted manifold
scat_man = ax_spec.scatter([], [], color="cyan", s=2.0, alpha=0.9, label=r"Manifold $\partial_x \arg(u)$")
ax_spec.legend(loc="upper left", fontsize=10)

# ==========================================
# 5. ANIMATION LOOP
# ==========================================
# Map solver frames to animation frames
frames_to_animate = len(solver.frames)
step = max(1, frames_to_animate // 200) # Limit animation to ~200 frames for smooth GIF

def update_nlse(frame_idx):
    u_current = solver.frames[frame_idx * step]
    t_current = frame_idx * step * solver.dt
    
    # 1. Update Physical Field
    line_u.set_data(solver.x_grid, np.real(u_current))
    
    # 2. Update Spectrogram
    spec_mag = np.abs(SFT.stft(u_current))
    t_stft = SFT.t(Nx) - Lx/2
    xi_stft = 2 * np.pi * SFT.f
    im_spec.set_data(spec_mag)
    im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
    im_spec.set_clim(vmin=0, vmax=spec_mag.max() * 0.7)
    
    # 3. Extract & Plot the Nonlinear Manifold
    xi_inst, mask = compute_instantaneous_manifold(u_current, dx, method='madelung', threshold=0.05)
    
    # Downsample for scatter plot performance
    ds_step = 4
    scat_man.set_offsets(np.c_[solver.x_grid[::ds_step][mask[::ds_step]], xi_inst[::ds_step][mask[::ds_step]]])
    
    ax_field.set_title(rf"True Nonlinear Manifold Interaction (NLSE) | $t = {t_current:.2f}$ | Kerr $\chi = {chi}$")
    
    return line_u, im_spec, scat_man

anim_nlse = FuncAnimation(fig, update_nlse, frames=len(solver.frames) // step, interval=60)
anim_nlse.save("19_nlse_manifold_collision_mad.gif", writer="pillow", fps=15)
plt.show()

In [ ]:
# ==========================================
# 5. ANIMATION LOOP
# ==========================================
frames_to_animate = len(solver.frames)
step = max(1, frames_to_animate // 200)

def update_colliding_airy_nlse(frame_idx):
    u_current = solver.frames[frame_idx * step]
    t_current = frame_idx * step * solver.dt
    
    # Update Physical Wavefield
    line_u.set_data(solver.x_grid, np.real(u_current))
    
    # Update Spectrogram Background
    spec_mag = np.abs(SFT.stft(u_current))
    t_stft = SFT.t(Nx) - Lx/2
    xi_stft = 2 * np.pi * SFT.f
    im_spec.set_data(spec_mag)
    im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
    im_spec.set_clim(vmin=0, vmax=spec_mag.max() * 0.7)
    
    # Dynamically Extract & Plot Hydrodynamic Manifold
    x_pts, xi_pts = compute_instantaneous_manifold(u_current, dx, method='reassigned', SFT=SFT, Lx=Lx, xi_lim=6.0)
    scat_man.set_offsets(np.c_[x_pts, xi_pts])
    
    ax_field.set_title(rf"Nonlinear Colliding Airy Beams (NLSE) | $t = {t_current:.2f}$ | $\chi = {chi}$")
    return line_u, im_spec, scat_man

anim = FuncAnimation(fig, update_colliding_airy_nlse, frames=len(solver.frames) // step, interval=60)
anim.save("19_nlse_manifold_collision_rea.gif", writer="pillow", fps=15)
plt.show()

In [ ]:
# ==========================================
# 5. ANIMATION LOOP
# ==========================================
frames_to_animate = len(solver.frames)
step = max(1, frames_to_animate // 200)

def update_colliding_airy_nlse(frame_idx):
    u_current = solver.frames[frame_idx * step]
    t_current = frame_idx * step * solver.dt
    
    # Update Physical Wavefield
    line_u.set_data(solver.x_grid, np.real(u_current))
    
    # Update Spectrogram Background
    spec_mag = np.abs(SFT.stft(u_current))
    t_stft = SFT.t(Nx) - Lx/2
    xi_stft = 2 * np.pi * SFT.f
    im_spec.set_data(spec_mag)
    im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
    im_spec.set_clim(vmin=0, vmax=spec_mag.max() * 0.7)
    
    # Dynamically Extract & Plot Hydrodynamic Manifold
    xi_inst, mask = compute_instantaneous_manifold(u_current, dx, method='envelope', threshold=0.04)

    ds_step = 4
    scat_man.set_offsets(np.c_[solver.x_grid[::ds_step][mask[::ds_step]], xi_inst[::ds_step][mask[::ds_step]]])
    
    ax_field.set_title(rf"Nonlinear Colliding Airy Beams (NLSE) | $t = {t_current:.2f}$ | $\chi = {chi}$")
    return line_u, im_spec, scat_man

anim = FuncAnimation(fig, update_colliding_airy_nlse, frames=len(solver.frames) // step, interval=60)
anim.save("19_nlse_manifold_collision_env.gif", writer="pillow", fps=15)
plt.show()

### 20. Nonlinear Colliding Airy Beams & Fold Caustic Dynamics (NLSE)

* **Mechanism:** Integration of the 1D Non-Linear Schrödinger Equation (NLSE) $\mathrm{i} u_t + u_{xx} + \chi \vert{}u\vert{}^2 u = 0$ via a pseudospectral solver. Two counter-propagating, exponentially truncated $A_2$ Airy fold caustics initialized at $x_0 = \pm 20.0$ with opposing momentum channels ($k_0 = \pm 2.0$) undergo direct collision under Kerr non-linearity ($\chi = 2.5$), driving strong self-phase modulation and Cross-Phase Modulation (XPM).
* **Phase Space:** Dynamic extraction of the evolving 1D Lagrangian manifold $\xi(x,t) = \operatorname{Im}\left(\frac{\partial_x u}{u}\right)$ using the Madelung hydrodynamic identity and spectral differentiation. Superimposed over a Short-Time Fourier Transform (STFT) spectrogram, it directly resolves the non-linear warping of parabolic caustic branches, localized momentum-space repulsion, and transient interference lattices generated during the interaction of self-accelerating wavepackets.

In [ ]:
# ==========================================
# 1. DEFINE THE NONLINEAR SCHRÖDINGER EQUATION (NLSE)
# ==========================================
# i * u_t + u_xx + chi * |u|^2 * u = 0
t, x = symbols('t x', real=True)
u = Function('u')(t, x)
chi = 2.5  # Kerr non-linearity strength

eq = Eq(
    diff(u, t), 
    1j * diff(u, x, 2) + 1j * chi * u * Abs(u)**2
)

# Initialize the ETD-RK4 Pseudospectral Solver
solver = PDESolver(eq, time_scheme='ETD-RK4', dealiasing_ratio=2/3, compute_energy=True)

# ==========================================
# 2. SETUP GRID & INITIAL CONDITIONS (COLLIDING AIRY BEAMS)
# ==========================================
Lx, Nx = 100.0, 2048
Lt, Nt = 12.0, 3000

x1_0, x2_0 = -35.0, 35.0   # Initial wavepacket centers
k0 = 2.0                 # Opposing momentum channels
decay = 0.08             # Exponential truncation factor

def initial_condition(x_arr):
    # 1. Right-moving Airy beam starting at x = -20.0
    # Head at x = -20, tail extends to the left (x < -20), moving right (+k0)
    s_right = 0.8 * (x_arr - x1_0)
    ai_right, _, _, _ = airy(s_right)
    env_right = np.exp(decay * (x_arr - x1_0))  # Damps tail as x -> -infinity
    u_right = ai_right * env_right * np.exp(1j * k0 * x_arr)

    # 2. Left-moving Airy beam starting at x = +20.0
    # Head at x = +20, tail extends to the right (x > +20), moving left (-k0)
    s_left = 0.8 * (x2_0 - x_arr)
    ai_left, _, _, _ = airy(s_left)
    env_left = np.exp(decay * (x2_0 - x_arr))   # Damps tail as x -> +infinity
    u_left = ai_left * env_left * np.exp(-1j * k0 * x_arr)

    return u_right + u_left

solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition, plot=True)

# ==========================================
# 3. SOLVE THE PDE (Pre-computation)
# ==========================================
print("\nSolving NLSE with colliding Airy beams...")
solver.solve()
print("Solving complete. Preparing animation...")


# ==========================================
# 4. STFT & VISUALIZATION SETUP
# ==========================================
dx = Lx / Nx
SFT = ShortTimeFFT(gaussian(256, std=32), hop=8, fs=1/dx, scale_to='magnitude', fft_mode='centered')

fig, (ax_field, ax_spec) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.5]})

line_u, = ax_field.plot([], [], color="tab:blue", lw=1.5, label=r"Field $\mathrm{Re}(u(x,t))$")
ax_field.set_ylabel("Amplitude")
ax_field.set_ylim(-2.0, 2.0)
ax_field.grid(True)
ax_field.legend(loc="upper left", fontsize=10)

im_spec = ax_spec.imshow(np.zeros((SFT.f_pts, SFT.p_num(Nx))), aspect='auto', origin='lower', cmap='magma')
ax_spec.set_xlabel("Spatial Coordinate $x$")
ax_spec.set_ylabel(r"Instantaneous Wavenumber $\xi(x,t)$")
ax_spec.set_ylim(-6.0, 6.0)

scat_man = ax_spec.scatter([], [], color="cyan", s=2.0, alpha=0.9, label=r"Lagrangian Manifold $\operatorname{Im}(\partial_x u / u)$")
ax_spec.legend(loc="upper left", fontsize=10)

# ==========================================
# 5. ANIMATION LOOP
# ==========================================
frames_to_animate = len(solver.frames)
step = max(1, frames_to_animate // 200)

def update_colliding_airy_nlse(frame_idx):
    u_current = solver.frames[frame_idx * step]
    t_current = frame_idx * step * solver.dt
    
    # Update Physical Wavefield
    line_u.set_data(solver.x_grid, np.real(u_current))
    
    # Update Spectrogram Background
    spec_mag = np.abs(SFT.stft(u_current))
    t_stft = SFT.t(Nx) - Lx/2
    xi_stft = 2 * np.pi * SFT.f
    im_spec.set_data(spec_mag)
    im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
    im_spec.set_clim(vmin=0, vmax=spec_mag.max() * 0.7)
    
    # Dynamically Extract & Plot Hydrodynamic Manifold
    xi_inst, mask = compute_instantaneous_manifold(u_current, dx, method='envelope', threshold=0.04)

    ds_step = 4
    scat_man.set_offsets(np.c_[solver.x_grid[::ds_step][mask[::ds_step]], xi_inst[::ds_step][mask[::ds_step]]])
    
    ax_field.set_title(rf"Nonlinear Colliding Airy Beams (NLSE) | $t = {t_current:.2f}$ | $\chi = {chi}$")
    return line_u, im_spec, scat_man

anim = FuncAnimation(fig, update_colliding_airy_nlse, frames=len(solver.frames) // step, interval=60)
anim.save("20_nlse_colliding_airy_env.gif", writer="pillow", fps=15)
plt.show()

In [ ]:
# ==========================================
# 5. ANIMATION LOOP
# ==========================================
frames_to_animate = len(solver.frames)
step = max(1, frames_to_animate // 200)

def update_colliding_airy_nlse(frame_idx):
    u_current = solver.frames[frame_idx * step]
    t_current = frame_idx * step * solver.dt
    
    # Update Physical Wavefield
    line_u.set_data(solver.x_grid, np.real(u_current))
    
    # Update Spectrogram Background
    spec_mag = np.abs(SFT.stft(u_current))
    t_stft = SFT.t(Nx) - Lx/2
    xi_stft = 2 * np.pi * SFT.f
    im_spec.set_data(spec_mag)
    im_spec.set_extent([t_stft[0], t_stft[-1], xi_stft[0], xi_stft[-1]])
    im_spec.set_clim(vmin=0, vmax=spec_mag.max() * 0.7)
    
    # Dynamically Extract & Plot Hydrodynamic Manifold
    x_pts, xi_pts = compute_instantaneous_manifold(u_current, dx, method='reassigned', SFT=SFT, Lx=Lx, xi_lim=6.0)
    scat_man.set_offsets(np.c_[x_pts, xi_pts])
    
    ax_field.set_title(rf"Nonlinear Colliding Airy Beams (NLSE) | $t = {t_current:.2f}$ | $\chi = {chi}$")
    return line_u, im_spec, scat_man

anim = FuncAnimation(fig, update_colliding_airy_nlse, frames=len(solver.frames) // step, interval=60)
anim.save("20_nlse_colliding_airy_rea.gif", writer="pillow", fps=15)
plt.show()

In [ ]:
# ==========================================================
# Kerr vs. linear control experiment (based on notebook cell 41)
# Paste into a cell AFTER `from imports import *` and the solver import.
# Same grid / beams / dt for every run; only the physics changes.
# ==========================================================

# ---------- shared setup (identical to cell 41) ----------
Lx, Nx = 80.0, 2048
Lt, Nt = 7.5, 2000          # same dt = 0.00375 as before; stops before wraparound
w0, k0 = 3.0, 2.5
x1_0, x2_0 = -20.0, 20.0
dx = Lx / Nx

def ic_two(x_arr):
    u1 = np.exp(-(x_arr - x1_0)**2 / w0**2 + 1j * k0 * x_arr)
    u2 = np.exp(-(x_arr - x2_0)**2 / w0**2 - 1j * k0 * x_arr)
    return u1 + u2

def ic_single(x_arr):                    # right-moving beam only -> pure SPM
    return np.exp(-(x_arr - x1_0)**2 / w0**2 + 1j * k0 * x_arr)

def run(chi, ic, label, Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt):
    t, x = symbols('t x', real=True)
    u = Function('u')(t, x)
    c = max(chi, 1e-12)                  # avoid sympy dropping the term when chi == 0
    eq = Eq(diff(u, t), 1j * diff(u, x, 2) + 1j * c * u * Abs(u)**2)
    s = PDESolver(eq, time_scheme='ETD-RK4', dealiasing_ratio=2/3, compute_energy=False)
    s.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=ic, plot=False)
    s.solve()
    frames = np.array(s.frames)
    times = np.linspace(0.0, Lt, len(frames))     # assumes uniform storage 0..Lt
    print(f"[{label}] chi={chi}: {len(frames)} frames, dt_frame={times[1]-times[0]:.4f}")
    return dict(label=label, chi=chi, x=np.asarray(s.x_grid), frames=frames, t=times)

runs = {
    'lin':    run(0.0, ic_two,    'A: two beams, chi=0'),
    'weak':   run(0.3, ic_two,    'D: two beams, chi=0.3'),
    'kerr':   run(3.0, ic_two,    'C: two beams, chi=3'),
    'single': run(3.0, ic_single, 'B: one beam, chi=3 (SPM only)'),
}

# ---------- diagnostics ----------
def diagnostics(r):
    x, frames, chi = r['x'], r['frames'], r['chi']
    k = 2 * np.pi * np.fft.fftfreq(len(x), d=x[1] - x[0])
    keys = ['mass', 'peak', 'P', 'H', 'leak',
            'm_p', 'xc_p', 'w_p', 'pk_p', 'm_m', 'xc_m', 'w_m', 'pk_m']
    D = {key: [] for key in keys}
    for u in frames:
        uh = np.fft.fft(u)
        ux = np.fft.ifft(1j * k * uh)
        rho = np.abs(u)**2
        D['mass'].append(rho.sum() * dx)
        D['peak'].append(np.abs(u).max())
        D['P'].append(np.imag(np.conj(u) * ux).sum() * dx)
        D['H'].append((np.abs(ux)**2 - 0.5 * chi * rho**2).sum() * dx)
        D['leak'].append((np.abs(uh)**2)[np.abs(k) > 5.0].sum() / (np.abs(uh)**2).sum())
        # split into +k / -k channels: separates the beams even while they overlap
        for tag, sel in (('p', k > 0), ('m', k < 0)):
            us = np.fft.ifft(uh * sel)
            rs = np.abs(us)**2
            m = rs.sum() * dx
            xc = (x * rs).sum() * dx / max(m, 1e-30)
            D['m_' + tag].append(m)
            D['xc_' + tag].append(xc)
            D['w_' + tag].append(np.sqrt(((x - xc)**2 * rs).sum() * dx / max(m, 1e-30)))
            D['pk_' + tag].append(np.abs(us).max())
    return {key: np.array(v) for key, v in D.items()}

for r in runs.values():
    r['diag'] = diagnostics(r)

# ---------- Figure 1: diagnostics time series ----------
fig1, axs = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
styles = {'lin': ('k', '-'), 'weak': ('tab:green', '--'),
          'kerr': ('tab:red', '-'), 'single': ('tab:blue', ':')}
def plot_all(ax, key, title, ylab=None, logy=False, relative=False):
    for name, r in runs.items():
        y = r['diag'][key]
        if relative:
            y = y / y[0] - 1.0
        c, ls = styles[name]
        ax.plot(r['t'], y, color=c, ls=ls, lw=1.6, label=r['label'])
    ax.set_title(title); ax.grid(True)
    if ylab: ax.set_ylabel(ylab)
    if logy: ax.set_yscale('log')

plot_all(axs[0, 0], 'peak',  r'peak $|u|$')
plot_all(axs[0, 1], 'w_p',   r'rms width, $+k$ beam')
plot_all(axs[0, 2], 'xc_p',  r'centroid, $+k$ beam (XPM shows as deviation from $\chi=0$)')
plot_all(axs[1, 0], 'm_p',   r'mass in $+k$ channel (exchange = nonlinear)')
plot_all(axs[1, 1], 'leak',  r'spectral leakage $|k|>5$ (four-wave mixing)', logy=True)
plot_all(axs[1, 2], 'H',     r'Hamiltonian drift (solver check)', relative=True)
for ax in axs[1]:
    ax.set_xlabel('t')
axs[0, 0].legend(fontsize=8)
for ax in axs.ravel():
    ax.axvline(4.0, color='gray', lw=0.8, ls=':')     # nominal collision time 2*20/(2*2.5*2)
fig1.tight_layout()
plt.show()

# ---------- Figure 2: reassigned manifold, chi=0 vs chi=3, before/during/after ----------
SFT = ShortTimeFFT(gaussian(256, std=32), hop=8, fs=1/dx, scale_to='magnitude', fft_mode='centered')
times_show = [2.0, 4.0, 6.0]
XI = 10.0

def frame_at(r, t_target):
    return int(round(t_target / r['t'][-1] * (len(r['t']) - 1)))

ref = np.abs(SFT.stft(runs['lin']['frames'][0])).max()   # common dB reference
fig2, axs2 = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)
for row, name in enumerate(['lin', 'kerr']):
    r = runs[name]
    for col, tt in enumerate(times_show):
        u = r['frames'][frame_at(r, tt)]
        mag = np.abs(SFT.stft(u))
        db = 20 * np.log10(np.maximum(mag / ref, 1e-6))
        xs = SFT.t(Nx) - Lx / 2
        xi = 2 * np.pi * SFT.f
        ax = axs2[row, col]
        ax.imshow(db, aspect='auto', origin='lower', cmap='magma', vmin=-50, vmax=0,
                  extent=[xs[0], xs[-1], xi[0], xi[-1]])
        xp, xip = compute_instantaneous_manifold(u, dx, method='reassigned', SFT=SFT,
                                                 Lx=Lx, xi_lim=XI, threshold=0.03)
        ax.scatter(xp, xip, s=1.2, c='cyan', alpha=0.5)
        ax.set_ylim(-XI, XI)
        ax.set_title(rf"{r['label']}  |  t = {tt:.1f}", fontsize=10)
        if col == 0: ax.set_ylabel(r'$\xi$')
        if row == 1: ax.set_xlabel('x')
fig2.tight_layout()
plt.show()

# ---------- Figure 3: difference field (nonlinear minus linear) ----------
fig3, axs3 = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for col, tt in enumerate(times_show):
    ul = runs['lin']['frames'][frame_at(runs['lin'], tt)]
    un = runs['kerr']['frames'][frame_at(runs['kerr'], tt)]
    axs3[col].plot(runs['lin']['x'], np.abs(ul), 'k', lw=1.2, label=r'$|u|$, $\chi=0$')
    axs3[col].plot(runs['kerr']['x'], np.abs(un), 'r', lw=1.2, label=r'$|u|$, $\chi=3$')
    axs3[col].set_title(f't = {tt:.1f}'); axs3[col].grid(True); axs3[col].set_xlabel('x')
axs3[0].legend(fontsize=8)
fig3.tight_layout()
plt.show()

# ---------- Sanity test: is the Kerr term actually applied? ----------
# For i u_t + u_xx + chi|u|^2 u = 0, u = sqrt(2/chi) sech(x) e^{it} is a stationary soliton.
# |u| must stay constant. If it disperses, the nonlinear term is not being applied as intended.
def soliton_test(chi=3.0, Lt_s=5.0, Nt_s=1334):
    ic = lambda x_arr: np.sqrt(2.0 / chi) / np.cosh(x_arr)
    r = run(chi, ic, 'soliton test', Lt=Lt_s, Nt=Nt_s)
    pk = np.array([np.abs(u).max() for u in r['frames']])
    print(f"soliton peak: start {pk[0]:.4f}, end {pk[-1]:.4f}, max drift {np.abs(pk/pk[0]-1).max():.2e}")
    return r

soliton_run = soliton_test()

# Optional: overlay from a linear run for contrast (should visibly disperse)
# lin_sol = run(0.0, lambda x: np.sqrt(2.0/3.0)/np.cosh(x), 'soliton, chi=0', Lt=5.0, Nt=1334)